In [2]:
# ===============================
# 1. Imports & Drive Mount
# ===============================
import pandas as pd
import numpy as np
from google.colab import drive

drive.mount('/content/drive')

# ===============================
# 2. Load CSV
# ===============================
file_path = "/content/drive/MyDrive/GEE_Forest_TimeSeries/TamilNadu_Forest_TimeSeries_Master.csv"
df = pd.read_csv(file_path)



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
print(df.head())
print(df['year'].unique())
print(df.columns)
# Keep only one row per location-year (average if needed)
# Define feature columns
features = ['BSI', 'Forest', 'NBR', 'NDMI', 'NDVI']

# Now group properly
df = df.groupby(['latitude', 'longitude', 'year'])[features].mean().reset_index()
df = df.groupby(['latitude', 'longitude', 'year'])[features].mean().reset_index()
sequence_length = 4  # 2021–2024
X_raw = []
locations = []

# Make sure duplicates are removed
df_unique = df.groupby(['latitude', 'longitude', 'year'])[features].mean().reset_index()

for (lat, lon), group in df_unique.groupby(['latitude', 'longitude']):
    group = group.sort_values('year')
    if len(group) == sequence_length:
        X_raw.append(group[features].values)
        locations.append([lat, lon])

X_raw = np.array(X_raw)       # (samples, 4, 4)
locations = np.array(locations)

print("Raw input shape:", X_raw.shape)
# =====================================================
# 1. Imports
# =====================================================
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.metrics import classification_report

# =====================================================
# 2. Create Labels (NDVI drop & BSI rise)
# =====================================================
# X_raw shape: (samples, time_steps, features)
# locations shape: (samples, 2) -> latitude, longitude

ndvi_drop = X_raw[:, 0, 0] - X_raw[:, -1, 0]   # NDVI 2021-2024
bsi_rise  = X_raw[:, -1, 3] - X_raw[:, 0, 3]   # BSI 2024-2021

# Use 85th percentile thresholds (adjust if too small)
ndvi_thr = np.percentile(ndvi_drop, 85)
bsi_thr  = np.percentile(bsi_rise, 85)

y = ((ndvi_drop > ndvi_thr) & (bsi_rise > bsi_thr)).astype(int)

print("\nLabel distribution:")
print("No deforestation:", (y == 0).sum())
print("Deforestation:", (y == 1).sum())

# =====================================================
# 3. Train-Test Split (Safe for small datasets)
# =====================================================
min_class_count = np.min([np.sum(y==0), np.sum(y==1)])
if min_class_count < 2:
    print("WARNING: Tiny dataset, skipping stratified split.")
    X_train, X_test, y_train, y_test, loc_train, loc_test = (
        X_raw, X_raw, y, y, locations, locations
    )
else:
    X_train, X_test, y_train, y_test, loc_train, loc_test = train_test_split(
        X_raw, y, locations,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)

# =====================================================
# 4. Scaling (No data leakage)
# =====================================================
scaler = MinMaxScaler()

X_train_rs = X_train.reshape(-1, X_train.shape[-1])
X_test_rs  = X_test.reshape(-1, X_test.shape[-1])

X_train = scaler.fit_transform(X_train_rs).reshape(X_train.shape)
X_test  = scaler.transform(X_test_rs).reshape(X_test.shape)

print("Scaled train shape:", X_train.shape)

# =====================================================
# 5. LSTM Model
# =====================================================
model = Sequential([
    LSTM(64, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', 'Recall', 'Precision']
)

model.summary()

# =====================================================
# 6. Train Model (Use class weighting if enough samples)
# =====================================================
if min_class_count < 2:
    print("Skipping class weighting due to tiny dataset.")
    history = model.fit(
        X_train, y_train,
        validation_data=(X_test, y_test),
        epochs=20,
        batch_size=1,  # small batch for tiny data
        verbose=1
    )
else:
    class_weight = {0: 1, 1: 12}
    history = model.fit(
        X_train, y_train,
        validation_data=(X_test, y_test),
        epochs=20,
        batch_size=128,
        class_weight=class_weight,
        verbose=1
    )

# =====================================================
# 7. Evaluation
# =====================================================
y_prob = model.predict(X_test)
y_pred = (y_prob > 0.35).astype(int)  # recall-friendly threshold

print(classification_report(y_test, y_pred))

# =====================================================
# 8. Save Predictions (with locations)
# =====================================================
results_df = pd.DataFrame({
    'latitude': loc_test[:, 0],
    'longitude': loc_test[:, 1],
    'deforestation': y_pred.flatten()
})

print(results_df.head())

results_df.to_csv(
    '/content/drive/MyDrive/TN_Deforestation_Predictions.csv',
    index=False
)

print("✅ Saved: TN_Deforestation_Predictions.csv")

             system:index       BSI  Forest       NBR      NDMI      NDVI  \
0  000000000000000062b6_0 -0.182323       1  0.517426  0.212965  0.767369   
1  000000000000000062b6_0 -0.100985       1  0.319837  0.141631  0.449275   
2  000000000000000062b6_0 -0.082196       1  0.313648  0.118453  0.444329   
3  000000000000000062b6_0 -0.086801       1  0.309993  0.124865  0.452542   
4  000000000000000062b7_0 -0.197501       1  0.536848  0.232422  0.766912   

   year                                               .geo  longitude  \
0  2021  {"geodesic":false,"type":"Point","coordinates"...  77.144172   
1  2022  {"geodesic":false,"type":"Point","coordinates"...  77.144172   
2  2023  {"geodesic":false,"type":"Point","coordinates"...  77.144172   
3  2024  {"geodesic":false,"type":"Point","coordinates"...  77.144172   
4  2021  {"geodesic":false,"type":"Point","coordinates"...  77.145071   

   latitude  
0  8.994382  
1  8.994382  
2  8.994382  
3  8.994382  
4  8.994382  
[2021 2022 202

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        17,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,033 (78.25 KB)

 Trainable params: 20,033 (78.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 8s 17ms/step - Precision: 0.1868 - Recall: 0.9549 - accuracy: 0.4327 - loss: 1.3142 - val_Precision: 0.2403 - val_Recall: 0.9716 - val_accuracy: 0.5815 - val_loss: 0.6729
Epoch 2/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - Precision: 0.3255 - Recall: 0.9583 - accuracy: 0.7262 - loss: 0.7822 - val_Precision: 0.3677 - val_Recall: 0.9901 - val_accuracy: 0.7688 - val_loss: 0.4331
Epoch 3/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - Precision: 0.4342 - Recall: 0.9738 - accuracy: 0.8252 - loss: 0.5573 - val_Precision: 0.5220 - val_Recall: 0.9827 - val_accuracy: 0.8762 - val_loss: 0.2397
Epoch 4/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - Precision: 0.5223 - Recall: 0.9849 - accuracy: 0.8764 - loss: 0.4038 - val_Precision: 0.7018 - val_Recall: 0.9790 - val_accuracy: 0.9410 - val_loss: 0.1419
Epoch 5/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - Precision: 0.6031 - Recall: 0.9877 - accuracy: 0.9106 - loss: 0.3029 - val_Precision: 0.7322 - val_Recall

In [5]:
!pip install folium
import pandas as pd
import folium
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
import numpy as np

# 1️⃣ Load deforestation points
df_def = pd.read_csv('/content/drive/MyDrive/TN_Deforestation_Predictions.csv')
df_deforest = df_def[df_def['deforestation']==1]

# 2️⃣ Load time-series indices
df_ts = pd.read_csv('/content/drive/MyDrive/GEE_Forest_TimeSeries/TamilNadu_Forest_TimeSeries_Master.csv')

# 3️⃣ Sort and compute changes
df_ts_sorted = df_ts.sort_values(by=['latitude','longitude','year'])

# First and last year values per point
first_year = df_ts_sorted.groupby(['latitude','longitude']).first().reset_index()
last_year  = df_ts_sorted.groupby(['latitude','longitude']).last().reset_index()

df_change = pd.merge(first_year, last_year, on=['latitude','longitude'], suffixes=('_start','_end'))

# Compute delta indices
df_change['NDVI_change'] = df_change['NDVI_end'] - df_change['NDVI_start']
df_change['NBR_change']  = df_change['NBR_end']  - df_change['NBR_start']
df_change['BSI_change']  = df_change['BSI_end']  - df_change['BSI_start']
df_change['NDMI_change'] = df_change['NDMI_end'] - df_change['NDMI_start']


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
# ==============================
# THRESHOLD FUNCTION
# ==============================
def get_thresholds(series):

    series = series.dropna()

    mean = series.mean()
    std  = series.std()
    q1   = series.quantile(0.25)
    q3   = series.quantile(0.75)

    return {
        'mean': mean,
        'std': std,
        'q1': q1,
        'q3': q3,
        'lower_2std': mean - 2*std,
        'upper_2std': mean + 2*std
    }

# ==============================
# COMPUTE THRESHOLDS
# ==============================
ndvi_thresh = get_thresholds(df_change['NDVI_change'])
nbr_thresh  = get_thresholds(df_change['NBR_change'])
bsi_thresh  = get_thresholds(df_change['BSI_change'])
ndmi_thresh = get_thresholds(df_change['NDMI_change'])

print("NDVI:", ndvi_thresh)
print("NBR :", nbr_thresh)
print("BSI :", bsi_thresh)
print("NDMI:", ndmi_thresh)

# ==============================
# MERGE DATA
# ==============================
df_merged = pd.merge(
    df_deforest,
    df_change[['latitude','longitude',
               'NDVI_change','NBR_change',
               'BSI_change','NDMI_change']],
    on=['latitude','longitude'],
    how='left'
)

# ==============================
# ADAPTIVE CLASSIFIER
# ==============================
def classify_cause_adaptive(row):

    ndvi = row['NDVI_change']
    nbr  = row['NBR_change']
    bsi  = row['BSI_change']

    if pd.isna(ndvi) or pd.isna(nbr) or pd.isna(bsi):
        return "Unknown"

    if nbr < nbr_thresh['lower_2std']:
        return "Fire"

    elif ndvi < ndvi_thresh['q1'] and bsi > bsi_thresh['q3']:
        return "Logging"

    elif ndvi < ndvi_thresh['mean'] and bsi > bsi_thresh['q1']:
        return "Agriculture"

    else:
        return "Urban/Other"

# ==============================
# APPLY CLASSIFICATION
# ==============================
df_merged['cause'] = df_merged.apply(classify_cause_adaptive, axis=1)

# ==============================
# SAVE OUTPUT
# ==============================
df_merged.to_csv(
    '/content/drive/MyDrive/TN_Deforestation_Causes_Adaptive.csv',
    index=False
)

# ==============================
# SUMMARY
# ==============================
print("\nDeforestation Cause Summary:")
print(df_merged['cause'].value_counts())


import pandas as pd
import folium
from folium.plugins import MarkerCluster

# ===========================
# Tamil Nadu map
# Approx latitude: 8–13, longitude: 76–80
# ===========================
m = folium.Map(location=[11.1, 78.6], zoom_start=7)

df = pd.read_csv('/content/drive/MyDrive/TN_Deforestation_Causes_Adaptive.csv')

print(df['cause'].value_counts())

# Define colors for causes
cause_colors = {
    'Fire': 'red',
    'Logging': 'orange',
    'Agriculture': 'yellow',
    'Urban/Other': 'gray'
}

marker_cluster = MarkerCluster().add_to(m)

for _, row in df.iterrows():
    color = cause_colors.get(row['cause'], 'blue')

    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=3,
        color=color,
        fill=True,
        fill_opacity=0.7,
    ).add_to(marker_cluster)

# Save map
m.save('/content/drive/MyDrive/tn_adaptive.html')

print("✅ Tamil Nadu map saved successfully!")

NDVI: {'mean': np.float64(-0.16751523074070834), 'std': 0.10965343607237009, 'q1': np.float64(-0.2423575925), 'q3': np.float64(-0.08911268250000001), 'lower_2std': np.float64(-0.3868221028854485), 'upper_2std': np.float64(0.051791641404031835)}
NBR : {'mean': np.float64(-0.11094423569831657), 'std': 0.1322079900071495, 'q1': np.float64(-0.1926299775), 'q3': np.float64(-0.024758325249999998), 'lower_2std': np.float64(-0.3753602157126156), 'upper_2std': np.float64(0.15347174431598243)}
BSI : {'mean': np.float64(0.02542046754910951), 'std': 0.08549605343108485, 'q1': np.float64(-0.028495068406114946), 'q3': np.float64(0.07840941215289587), 'lower_2std': np.float64(-0.1455716393130602), 'upper_2std': np.float64(0.1964125744112792)}
NDMI: {'mean': np.float64(-0.03848384904800983), 'std': 0.09882654347987382, 'q1': np.float64(-0.0956455715), 'q3': np.float64(0.01839249055), 'lower_2std': np.float64(-0.23613693600775748), 'upper_2std': np.float64(0.15916923791173782)}

Deforestation Cause Sum

In [8]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# ==============================
# LOAD DEFORESTATION PREDICTIONS
# ==============================
df_def = pd.read_csv(
    '/content/drive/MyDrive/TN_Deforestation_Predictions.csv'
)

print("Total samples:", len(df_def))
print(df_def.head())

# Filter only deforestation = 1
df_deforest = df_def[df_def['deforestation'] == 1].copy()
print("Total deforestation samples:", len(df_deforest))

# ==============================
# LOAD CAUSE DATA
# ==============================
df_cause = pd.read_csv(
    '/content/drive/MyDrive/TN_Deforestation_Causes_Adaptive.csv'
)

print(df_cause.head())

# Ensure latitude/longitude are float to avoid merge issues
df_deforest['latitude'] = df_deforest['latitude'].astype(float)
df_deforest['longitude'] = df_deforest['longitude'].astype(float)
df_cause['latitude'] = df_cause['latitude'].astype(float)
df_cause['longitude'] = df_cause['longitude'].astype(float)

# ==============================
# LOAD TAMIL NADU DISTRICTS
# ==============================
tn = gpd.read_file('/content/drive/MyDrive/tn.geojson')

# CRS SAFETY
if tn.crs is None:
    tn.set_crs("EPSG:4326", inplace=True)

tn = tn.to_crs("EPSG:4326")
tn["state"] = "Tamil Nadu"

print(tn.head())

# ==============================
# MERGE CAUSE DATA
# ==============================
df_deforest = df_deforest.merge(
    df_cause[['latitude', 'longitude', 'cause']],
    on=['latitude', 'longitude'],
    how='left'
)

print(df_deforest.head())

# ==============================
# CREATE POINT GEOMETRY
# ==============================
geometry = [Point(xy) for xy in zip(df_deforest['longitude'], df_deforest['latitude'])]

gdf_points = gpd.GeoDataFrame(
    df_deforest,
    geometry=geometry,
    crs="EPSG:4326"
)

print(gdf_points.head())

Total samples: 6000
    latitude  longitude  deforestation
0  13.285634  80.107715              0
1  13.104174  80.049324              0
2  13.282939  80.111308              0
3  13.247905  80.070884              0
4  13.445534  80.159817              1
Total deforestation samples: 1252
    latitude  longitude  deforestation  NDVI_change  NBR_change  BSI_change  \
0  13.445534  80.159817              1    -0.046537    0.154859   -0.153358   
1  13.392533  80.154427              1    -0.066606    0.056518   -0.098742   
2  13.446432  80.160715              1     0.022941    0.196240   -0.188392   
3   9.924138  79.115076              1    -0.020058    0.014173   -0.104667   
4   9.922341  79.062076              1    -0.085529    0.002321   -0.044267   

   NDMI_change        cause  
0     0.171005  Urban/Other  
1     0.094005  Urban/Other  
2     0.199479  Urban/Other  
3     0.104244  Urban/Other  
4     0.037443  Urban/Other  
   ID_0  ISO NAME_0  ID_1      NAME_1  ID_2              

In [10]:
import geopandas as gpd

# ==============================
# SPATIAL JOIN (Points → Districts)
# ==============================
gdf_joined = gpd.sjoin(
    gdf_points,
    tn,                # Tamil Nadu GeoDataFrame
    how='left',
    predicate='intersects'
)

print(gdf_joined.head())

print("Total joined points:", len(gdf_joined))
print("Points without district:", gdf_joined['NAME_2'].isna().sum())
print(gdf_joined['NAME_2'].value_counts())

# ==============================
# DISTRICT SUMMARY
# ==============================
district_summary = (
    gdf_joined
    .groupby('NAME_2')
    .size()
    .reset_index(name='deforestation_points')
    .sort_values(by='deforestation_points', ascending=False)
)

print(district_summary)

district_summary.to_csv(
    '/content/drive/MyDrive/TN_District_Wise_Deforestation.csv',
    index=False
)

print("Saved Tamil Nadu district summary")

# ==============================
# DISTRICT × CAUSE SUMMARY
# ==============================
cause_summary = (
    gdf_joined
    .groupby(['NAME_2', 'cause'])
    .size()
    .reset_index(name='count')
)

print(cause_summary.head())

cause_summary.to_csv(
    '/content/drive/MyDrive/TN_District_Wise_Deforestation_Causes.csv',
    index=False
)

print("Saved Tamil Nadu cause summary")

    latitude  longitude  deforestation        cause  \
0  13.445534  80.159817              1  Urban/Other   
1  13.392533  80.154427              1  Urban/Other   
2  13.446432  80.160715              1  Urban/Other   
3   9.924138  79.115076              1  Urban/Other   
4   9.922341  79.062076              1  Urban/Other   

                    geometry  index_right   ID_0  ISO NAME_0  ID_1  \
0  POINT (80.15982 13.44553)         28.0  105.0  IND  India  31.0   
1  POINT (80.15443 13.39253)         28.0  105.0  IND  India  31.0   
2  POINT (80.16072 13.44643)         28.0  105.0  IND  India  31.0   
3   POINT (79.11508 9.92414)         22.0  105.0  IND  India  31.0   
4   POINT (79.06208 9.92234)         22.0  105.0  IND  India  31.0   

       NAME_1   ID_2       NAME_2 NL_NAME_2   VARNAME_2    TYPE_2 ENGTYPE_2  \
0  Tamil Nadu  480.0  Thiruvallur      None        None  District  District   
1  Tamil Nadu  480.0  Thiruvallur      None        None  District  District   
2  Tamil Na

In [12]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import folium
import numpy as np

# =====================================================
# STEP 2: Load Tamil Nadu District Boundaries
# =====================================================
tn = gpd.read_file('/content/drive/MyDrive/tn.geojson')

# CRS safety
if tn.crs is None:
    tn.set_crs(epsg=4326, inplace=True)

tn = tn.to_crs(epsg=4326)
tn["state"] = "Tamil Nadu"
gdf_districts = tn.copy()

# =====================================================
# STEP 3: Load Deforestation Predictions
# =====================================================
df = pd.read_csv("/content/drive/MyDrive/TN_Deforestation_Predictions.csv")

# Ensure afforestation column exists
if 'afforestation' not in df.columns:
    df['afforestation'] = 0

# =====================================================
# STEP 4: Convert to GeoDataFrame
# =====================================================
gdf_points = gpd.GeoDataFrame(
    df,
    geometry=[Point(xy) for xy in zip(df.longitude, df.latitude)],
    crs="EPSG:4326"
)

# =====================================================
# STEP 5: Spatial Join
# =====================================================
points_with_district = gpd.sjoin(
    gdf_points,
    gdf_districts,
    how="left",
    predicate="within"
)

# =====================================================
# STEP 6: District Statistics
# =====================================================
district_total = (
    points_with_district
    .groupby("NAME_2")
    .size()
    .reset_index(name="total_samples")
)

district_deforestation = (
    points_with_district[points_with_district["deforestation"] == 1]
    .groupby("NAME_2")
    .size()
    .reset_index(name="deforestation_samples")
)

district_afforestation = (
    points_with_district[points_with_district["afforestation"] == 1]
    .groupby("NAME_2")
    .size()
    .reset_index(name="afforestation_samples")
)

# Merge
gdf_districts = gdf_districts.merge(district_total, on="NAME_2", how="left")
gdf_districts = gdf_districts.merge(district_deforestation, on="NAME_2", how="left")
gdf_districts = gdf_districts.merge(district_afforestation, on="NAME_2", how="left")

gdf_districts.fillna(0, inplace=True)

# =====================================================
# STEP 7: Area + Rate Calculations
# =====================================================
PIXEL_AREA_SQKM = 0.0001
PIXEL_AREA_HA = 0.01

gdf_districts["deforestation_rate_%"] = (
    gdf_districts["deforestation_samples"] /
    gdf_districts["total_samples"] * 100
).fillna(0)

gdf_districts["deforestation_area_sqkm"] = (
    gdf_districts["deforestation_samples"] * PIXEL_AREA_SQKM
)

gdf_districts["deforestation_area_ha"] = (
    gdf_districts["deforestation_samples"] * PIXEL_AREA_HA
)

gdf_districts["afforestation_rate_%"] = (
    gdf_districts["afforestation_samples"] /
    gdf_districts["total_samples"] * 100
).fillna(0)

gdf_districts["afforestation_area_sqkm"] = (
    gdf_districts["afforestation_samples"] * PIXEL_AREA_SQKM
)

gdf_districts["afforestation_area_ha"] = (
    gdf_districts["afforestation_samples"] * PIXEL_AREA_HA
)

# =====================================================
# STEP 8: Create Folium Map (center Tamil Nadu)
# =====================================================
m = folium.Map(location=[11.1, 78.6], zoom_start=7)

# =====================================================
# STEP 9: Choropleth
# =====================================================
min_val = gdf_districts["deforestation_samples"].min()
max_val = gdf_districts["deforestation_samples"].max()

if max_val == min_val:
    bins = [min_val, max_val + 1]
else:
    bins = np.linspace(min_val, max_val, num=6).tolist()

folium.Choropleth(
    geo_data=gdf_districts.to_json(),
    data=gdf_districts,
    columns=["NAME_2", "deforestation_samples"],
    key_on="feature.properties.NAME_2",
    fill_color="YlOrRd",
    bins=bins,
    fill_opacity=0.7,
    line_opacity=0.4,
    legend_name="Deforestation Samples (Tamil Nadu)"
).add_to(m)

# =====================================================
# STEP 10: Tooltips
# =====================================================
folium.GeoJson(
    gdf_districts,
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "NAME_2",
            "deforestation_samples",
            "deforestation_rate_%",
            "deforestation_area_sqkm",
            "deforestation_area_ha",
            "afforestation_samples",
            "afforestation_rate_%",
            "afforestation_area_sqkm",
            "afforestation_area_ha"
        ],
        aliases=[
            "District:",
            "Deforestation samples:",
            "Deforestation rate (%):",
            "Deforestation Area (sq.km):",
            "Deforestation Area (ha):",
            "Afforestation samples:",
            "Afforestation rate (%):",
            "Afforestation Area (sq.km):",
            "Afforestation Area (ha):"
        ],
        localize=True
    )
).add_to(m)

# =====================================================
# STEP 11: Alert Popup
# =====================================================
state_def = gdf_districts["deforestation_samples"].sum()

top3 = (
    gdf_districts.sort_values(by="deforestation_samples", ascending=False)
    .head(3)
)

top_districts_html = ""
for _, row in top3.iterrows():
    top_districts_html += f"• {row['NAME_2']} ({int(row['deforestation_samples'])})<br>"

popup_html = f"""
<b>🌳 Tamil Nadu Afforestation Alert</b><br><br>
Total Deforestation Points: <b>{int(state_def)}</b><br><br>
High Impact Districts:<br>
{top_districts_html}<br>
🌱 Immediate afforestation drives recommended.
"""

folium.Marker(
    location=[11.1, 78.6],
    popup=popup_html,
    icon=folium.Icon(color="green")
).add_to(m)

# =====================================================
# STEP 12: Save Map
# =====================================================
folium.LayerControl().add_to(m)
m.save("/content/drive/MyDrive/tn_forest.html")

print("✅ Tamil Nadu map saved successfully with afforestation recommendation!")

/tmp/ipykernel_386/1688716881.py:77: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  gdf_districts.fillna(0, inplace=True)


✅ Tamil Nadu map saved successfully with afforestation recommendation!
